## **Film Scraper**

L'obiettivo è quello di ottenere informazioni (genere, durata, immagine di copertina, ...) di film e serie tv attraverso tecniche di Web Scraping. 

### **Libraries**

In [ ]:
import json
import re
import urllib.parse
from pathlib import Path

import requests
from bs4 import BeautifulSoup

print('Libraries Imported')

### **Settings**

In [ ]:
# data
AVAILABLE_DATA = {
    1: 'sample.json',
    2016: 'movies_and_tv_series_2016.json',
    2017: 'movies_and_tv_series_2017.json',
    2018: 'movies_and_tv_series_2018.json',
    2019: 'movies_and_tv_series_2019.json',
    2020: 'movies_and_tv_series_2020.json',
    2021: 'movies_and_tv_series_2021.json',
    2022: 'movies_and_tv_series_2022.json',
    2023: 'movies_and_tv_series_2023.json',
}

# web scraping
BASE_URL_MOVIE = 'https://www.themoviedb.org/search/movie?query='
BASE_URL_TV_SERIES = 'https://www.themoviedb.org/search/tv?query='
HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/108.0.0.0 Safari/537.36'
    )
}

# settings
DATA = [
    # 2016,
    # 2017,
    # 2018,
    # 2019,
    # 2020,
    # 2021,
    # 2022,
    2023,
]

# DATA = [1]

print('Settings Configured')

### **Logic**

#### *Encoding*

In [ ]:
def encode_url(base_url, movie_title):
    """Encode the movie title and returns the full URL."""
    movie_title_encoded = urllib.parse.quote(movie_title)
    return base_url + movie_title_encoded


encode_url(BASE_URL_MOVIE, 'Jason Bourne')

#### *Scraping*

In [4]:
def scrape_data(data, exceptions=None):
    """Scraper function to extract movie data from The Movie Database."""
    # iterate over each title
    for k, v in data.items():
        print(f"\n📢 {k}: {v['title']} ({v['category']})")

        # 0. Check for exceptions
        title = v['title']
        if title in exceptions:
            print('⚠️ Exception: skipping scraping for this entry.')
            # override all values with the exception
            v['url'] = exceptions[title]['url']
            v['release_date'] = exceptions[title]['release_date']
            v['genres'] = exceptions[title]['genres']
            v['runtime'] = exceptions[title]['runtime']
            v['poster_url'] = exceptions[title]['poster_url']
            continue

        # 1. Setup based on category
        if v['category'] == 'Film':
            base_url = BASE_URL_MOVIE
            keyword = 'movie'
        elif v['category'] in ['Serie', 'Serie TV']:
            base_url = BASE_URL_TV_SERIES
            keyword = 'tv'
        else:
            print('❌ Invalid category. Using Movie Settings as default.')
            keyword = 'movie'
            base_url = BASE_URL_MOVIE

        # 2. Encode the URL, based on category

        # preprocess the title
        title = re.sub(r' S\d+', '', title)

        # encode the URL
        search_url = encode_url(base_url, title)

        # 3. Send a request (search page)
        try:
            response = requests.get(
                search_url,
                headers=HEADERS,
                timeout=60,
            )
            html_content = response.text
            print(f'✅ Request successful. URL: {search_url}')
        except Exception as e:  # noqa
            print(f'❌ An error occurred in the request. Details: {e}')
            continue

        # 4. Parse the HTML content
        soup = BeautifulSoup(html_content, 'html.parser')

        # This should find the first a with class="result" that references a movie link ("/movie/...").
        first_result = soup.select_one(f'a.result[href^="/{keyword}/"]')

        if first_result:
            relative_url = first_result.get('href')  # e.g. "/movie/241848-the-guest"
            # Construct a full URL to follow:
            full_url = 'https://www.themoviedb.org' + relative_url
            v['url'] = full_url
            print(f'🔗 Found movie URL: {full_url}')
        else:
            v['url'] = None
            print('❌ No movie found.')

        # 5. Send a request (movie page)
        if v['url']:
            try:
                response = requests.get(
                    v['url'],
                    headers=HEADERS,
                    timeout=60,
                )
                html_content = response.text
                print(f'✅ Request successful. URL: {v["url"]}')
            except Exception as e:  # noqa
                print(f'❌ An error occurred in the request. Details: {e}')
                continue

        # 6. Parse the HTML content
        soup = BeautifulSoup(html_content, 'html.parser')

        # --- 1. Release Date ---
        release_date = None
        if keyword == 'movie':
            release_date_elem = soup.select_one('span.release')
            if release_date_elem:
                release_date = release_date_elem.text.strip()
                release_date = release_date.split('(')[0].strip()  # -> "01/09/2016"
        else:
            release_date_elem = soup.select_one('span.release_date')
            if release_date_elem:
                release_date = release_date_elem.text.strip().strip('(').strip(')')

        # --- 2. Genre ---
        # <span class="genres">
        #   <a href="/genre/28-action/movie">Action</a>, <a href="/genre/53-thriller/movie">Thriller</a>
        # </span>
        genres_elem = soup.select_one('span.genres')
        genres = []
        if genres_elem:
            # collect the text from all <a> tags inside .genres
            genres = [a.text.strip() for a in genres_elem.select('a')]

        # --- 3. Runtime ---
        # <span class="runtime">2h 3m</span>
        runtime_elem = soup.select_one('span.runtime')
        runtime = runtime_elem.text.strip() if runtime_elem else None

        # --- 4. URL of the Poster Image ---
        # <img class="poster w-full" src="https://media.themoviedb.org/t/p/w300_and_h450_bestv2/ziU0b3hRM6raH1u4wym02EYMLZ6.jpg" ... >
        poster_elem = soup.select_one('#original_header img.poster')
        poster_url = poster_elem.get('src') if poster_elem else None

        # Store the extracted data back into your films dictionary
        v['release_date'] = release_date
        v['genres'] = genres
        v['runtime'] = runtime
        v['poster_url'] = poster_url

        print(f'📅 Release Date: {release_date}')
        print(f'🎭 Genres: {genres}')
        print(f'🕒 Runtime: {runtime}')
        print(f'🖼️ Poster URL: {poster_url}')

### **Let's Go**

Otteniamo le informazioni che ci servono, anno per anno.

In [5]:
exceptions = {
    'American Pie - Il primo assaggio non si scorda mai': {
        'url': 'https://www.themoviedb.org/movie/2105-american-pie',
        'release_date': '29/10/1999',
        'genres': ['Comedy', 'Romance'],
        'runtime': '1h 35m',
        'poster_url': 'https://media.themoviedb.org/t/p/w600_and_h900_bestv2/5P68by2Thn8wHAziyWGEw2O7hco.jpg',
    }
}

In [ ]:
for year in DATA:
    # import the data
    data_path = Path(f'src/custom_logics/movies_scraper/data/{AVAILABLE_DATA[year]}')
    with Path.open(data_path) as f:
        data = json.load(f)
    # start the scraping
    scrape_data(data, exceptions)
    # save the updated data
    res_path = Path(f'src/custom_logics/movies_scraper/results/{AVAILABLE_DATA[year]}')
    with Path.open(res_path, 'w') as f:
        json.dump(data, f, indent=2)